In [3]:
# For SIZE OF DATABENTO REQUESTS
# path is whatever under data/raw/some_id/batch.json

import json
from collections import defaultdict
from pathlib import Path

manifest = Path(
    "C:/Users/Adarsh Arun/Downloads/BookSpace/data/raw/5f5736543928343c/batch.json"
    )

with manifest.open() as f:
    data = json.load(f)

for job_name, job in data["jobs"].items():
    dbn_files = [
        f for f in job["files"]
        if f["filename"].endswith(".dbn.zst")
    ]

    total = sum(f["size"] for f in dbn_files)

    print(
        job_name,
        len(dbn_files),
        f"{total / 2**30:.3f} GiB",
    )

    monthly = defaultdict(int)

    for f in dbn_files:
        # glbx-mdp3-YYYYMMDD....
        date = f["filename"].split("-")[2].split(".")[0]
        monthly[date[:6]] += f["size"]

    for month, size in sorted(monthly.items()):
        print(
            "   ",
            month,
            f"{size / 2**30:.3f} GiB",
        )

definition 103 0.000 GiB
    202501 0.000 GiB
    202502 0.000 GiB
    202503 0.000 GiB
    202504 0.000 GiB
mbo 103 18.581 GiB
    202501 3.791 GiB
    202502 3.435 GiB
    202503 5.377 GiB
    202504 5.978 GiB


In [1]:
# not a parser but lolz

from pathlib import Path
import numpy as np
 
old = Path(r"data/processed/databento/pilot-fast-20250102")
new = Path(r"data/processed/databento/pilot-normal-20250102")
 
old_files = sorted(old.glob("rows-*.npz"))
new_files = sorted(new.glob("rows-*.npz"))
 
assert len(old_files) == len(new_files)
 
for a, b in zip(old_files, new_files):
    with np.load(a) as x, np.load(b) as y:
        assert x.files == y.files
 
        for key in x.files:
            if not np.array_equal(x[key], y[key]):
                diff = np.max(np.abs(x[key] - y[key])) if np.issubdtype(x[key].dtype, np.number) else None
                raise AssertionError(
                    f"{a.name} differs in {key}, max diff={diff}"
                )
 
print("IDENTICAL")

IDENTICAL


In [ ]:
# Check full databento data error it was just good friday.

from pathlib import Path
from datetime import datetime
from collections import defaultdict
from statistics import mean, median
import json


path = Path("C:/Users/Adarsh Arun/Downloads/BookSpace/data/processed/databento/daily/parallel-build-summary.json")

if path is None:
    raise FileNotFoundError("parallel-build-summary.json not found")

with path.open("r", encoding="utf-8") as f:
    d = json.load(f)

ok = d.get("ok", [])
failed = d.get("failed", [])
jobs = d["discovered_jobs"]
workers = d["workers"]
wall = d["wall_seconds"]

rows = sum(x["rows"] for x in ok)
replay_seconds = sum(x["elapsed_seconds"] for x in ok)
worker_seconds = sum(x["worker_wall_seconds"] for x in ok)

rates = [
    x["rows"] / x["elapsed_seconds"]
    for x in ok
    if x["elapsed_seconds"] > 0
]

months = defaultdict(lambda: {"days": 0, "rows": 0})
weekdays = defaultdict(lambda: {"days": 0, "rows": 0})

for x in ok:
    dt = datetime.fromisoformat(x["day"])

    month = dt.strftime("%Y-%m")
    months[month]["days"] += 1
    months[month]["rows"] += x["rows"]

    day = dt.strftime("%a")
    weekdays[day]["days"] += 1
    weekdays[day]["rows"] += x["rows"]

sunday_rows = weekdays["Sun"]["rows"]
sunday_days = weekdays["Sun"]["days"]
other_rows = rows - sunday_rows
other_days = len(ok) - sunday_days

sunday_avg = sunday_rows / sunday_days if sunday_days else 0
other_avg = other_rows / other_days if other_days else 0

def pct(a, b):
    return 100 * a / b if b else 0

def duration(seconds):
    seconds = round(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def n(x):
    return f"{x:,.0f}"

print("BUILD")
print(f"{'Discovered':<24} {jobs:>15,}")
print(f"{'Succeeded':<24} {len(ok):>7,} / {jobs:<7,} {pct(len(ok), jobs):>8.2f}%")
print(f"{'Failed':<24} {len(failed):>7,} / {jobs:<7,} {pct(len(failed), jobs):>8.2f}%")
print(f"{'Workers':<24} {workers:>15,}")
print(f"{'Wall time':<24} {duration(wall):>15}")
print()

print("ROWS")
print(f"{'Total':<24} {rows:>15,}")
print(f"{'Mean / file':<24} {mean(x['rows'] for x in ok):>15,.0f}")
print(f"{'Median / file':<24} {median(x['rows'] for x in ok):>15,.0f}")
print(f"{'Min / file':<24} {min(x['rows'] for x in ok):>15,}")
print(f"{'Max / file':<24} {max(x['rows'] for x in ok):>15,}")
print()

print("THROUGHPUT")
print(f"{'Aggregate':<24} {rows / wall:>12,.0f} rows/s")
print(f"{'Weighted / worker':<24} {rows / replay_seconds:>12,.0f} rows/s")
print(f"{'Median / worker':<24} {median(rates):>12,.0f} rows/s")
print(f"{'Worker utilisation':<24} {pct(worker_seconds, workers * wall):>14.2f}%")
print()

print("SUNDAY")
print(f"{'Files':<24} {sunday_days:>7,} / {len(ok):<7,} {pct(sunday_days, len(ok)):>8.2f}%")
print(f"{'Rows':<24} {sunday_rows:>15,}")
print(f"{'Row share':<24} {pct(sunday_rows, rows):>14.3f}%")
print(f"{'Sunday mean':<24} {sunday_avg:>15,.0f}")
print(f"{'Non-Sunday mean':<24} {other_avg:>15,.0f}")
print(f"{'Mean ratio':<24} {'1 : ' + f'{other_avg / sunday_avg:.2f}':>15}")
print()

print("MONTHS")
for month in sorted(months):
    x = months[month]
    print(
        f"{month:<10}"
        f"{x['days']:>5} files   "
        f"{x['rows']:>15,} rows   "
        f"{pct(x['rows'], rows):>7.2f}%"
    )

if failed:
    print()
    print("FAILURES")
    for x in failed:
        print(f"{x['day']:<12} {x['error']}")

BUILD
Discovered                           103
Succeeded                    102 / 103        99.03%
Failed                         1 / 103         0.97%
Workers                                6
Wall time                       02:17:11

ROWS
Total                      1,054,065,838
Mean / file                   10,333,979
Median / file                 10,855,866
Min / file                        41,727
Max / file                    28,725,459

THROUGHPUT
Aggregate                     128,057 rows/s
Weighted / worker              21,943 rows/s
Median / worker                21,937 rows/s
Worker utilisation                97.29%

SUNDAY
Files                         17 / 102        16.67%
Rows                           4,136,458
Row share                         0.392%
Sunday mean                      243,321
Non-Sunday mean               12,352,110
Mean ratio                     1 : 50.76

MONTHS
2025-01      27 files       215,746,059 rows     20.47%
2025-02      24 files       197,370,